<a href="https://www.kaggle.com/code/mariammouh/mini-project-cv?scriptVersionId=304112789" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Face Anti-Spoofing — Transfer Learning Strategies


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from sklearn.metrics import accuracy_score, precision_score, f1_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

## Shared Setup: Config, Dataset, Transforms & Helper Functions

In [ ]:
# CONFIG
BASE_PATH = "/kaggle/input/datasets/ahmedruhshan/lcc-fasd-casia-combined/lcc-fasd-casia/LCC_FASD"

CONFIG = {
    "train_dir"   : os.path.join(BASE_PATH, "LCC_FASD_training"),
    "val_dir"     : os.path.join(BASE_PATH, "LCC_FASD_development"),
    "test_dir"    : os.path.join(BASE_PATH, "LCC_FASD_evaluation"),
    "img_size"    : 224,
    "batch_size"  : 32,
    "epochs"      : 20,
    "lr"          : 1e-4,
    "weight_decay": 1e-4,
    "num_classes" : 2,
    "patience"    : 5,
    "device"      : "cuda" if torch.cuda.is_available() else "cpu",
    "seed"        : 42
}
print(f"Device: {CONFIG['device']}")


# DATASET
class FaceAntiSpoofDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.image_paths = []
        self.labels      = []
        self.transform   = transform

        real_dir = os.path.join(data_dir, "real")
        for fname in os.listdir(real_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                self.image_paths.append(os.path.join(real_dir, fname))
                self.labels.append(1)

        spoof_dir = os.path.join(data_dir, "spoof")
        for fname in os.listdir(spoof_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                self.image_paths.append(os.path.join(spoof_dir, fname))
                self.labels.append(0)

        print(f"  {os.path.basename(data_dir)} -> real: {self.labels.count(1)} | spoof: {self.labels.count(0)}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# TRANSFORMS
train_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# SAMPLER
def get_sampler(labels):
    class_counts   = np.bincount(labels)
    weights        = 1.0 / class_counts
    sample_weights = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_weights, len(sample_weights))


# TRAIN ONE EPOCH
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  Train"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


# EVALUATE
def evaluate(model, loader, criterion, device, split="Val"):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"  {split}"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    return total_loss / len(loader), acc, prec, f1, all_labels, all_preds


# GENERIC PIPELINE
def run_pipeline(build_fn, save_path, strategy_name):
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    torch.manual_seed(CONFIG["seed"])

    print(f"\n{'='*60}")
    print(f"  STRATEGY: {strategy_name}")
    print(f"{'='*60}")

    print("\nLoading datasets:")
    train_dataset = FaceAntiSpoofDataset(CONFIG["train_dir"], train_transform)
    val_dataset   = FaceAntiSpoofDataset(CONFIG["val_dir"],   val_transform)
    test_dataset  = FaceAntiSpoofDataset(CONFIG["test_dir"],  val_transform)

    sampler      = get_sampler(train_dataset.labels)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                              sampler=sampler, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                              shuffle=False,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                              shuffle=False,  num_workers=2, pin_memory=True)

    model = build_fn().to(CONFIG["device"])

    counts    = np.bincount(train_dataset.labels)
    weights   = torch.tensor(1.0 / counts, dtype=torch.float).to(CONFIG["device"])
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_f1 = 0
    epochs_no_improve = 0

    for epoch in range(CONFIG["epochs"]):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch+1}/{CONFIG['epochs']}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, CONFIG["device"])
        val_loss, val_acc, val_prec, val_f1, _, _ = evaluate(model, val_loader, criterion, CONFIG["device"], "Val")
        scheduler.step()
        print(f"  Train -> Loss: {train_loss:.4f} | Acc: {train_acc*100:.2f}%")
        print(f"  Val   -> Loss: {val_loss:.4f} | Acc: {val_acc*100:.2f}% | Precision: {val_prec*100:.2f}% | F1: {val_f1*100:.2f}%")
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            print(f"  Best model saved (F1={best_f1*100:.2f}%)")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{CONFIG['patience']})")
            if epochs_no_improve >= CONFIG["patience"]:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

    print("\nFinal evaluation on Test Set:")
    model.load_state_dict(torch.load(save_path))
    _, test_acc, test_prec, test_f1, true_labels, pred_labels = evaluate(
        model, test_loader, criterion, CONFIG["device"], "Test"
    )
    print(f"\nFinal Results ({strategy_name})")
    print(f"  Accuracy  : {test_acc*100:.2f}%")
    print(f"  Precision : {test_prec*100:.2f}%")
    print(f"  F1-Score  : {test_f1*100:.2f}%")

    # LOSS & ACCURACY CURVES + CONFUSION MATRIX
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"],   label="Val")
    axes[0].set_title(f"Loss — {strategy_name}")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"],   label="Val")
    axes[1].set_title(f"Accuracy — {strategy_name}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    cm = confusion_matrix(true_labels, pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Spoof", "Real"])
    disp.plot(ax=axes[2], colorbar=False, cmap="Blues")
    axes[2].set_title(f"Confusion Matrix — {strategy_name}")

    plt.tight_layout()
    safe_name = strategy_name.lower().replace(" ", "_").replace("-", "_")
    plt.savefig(f"/kaggle/working/curves_{safe_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Curves saved: curves_{safe_name}.png")

    return {"strategy": strategy_name, "accuracy": test_acc*100,
            "precision": test_prec*100, "f1": test_f1*100}

print("All shared functions loaded.")

## Strategy 1 — Freeze ALL
All layers frozen. Only the classification head is trainable.

In [ ]:
# MODEL — FREEZE ALL (head only)
def build_model_freeze_all():
    model = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=True,
        num_classes=CONFIG["num_classes"]
    )
    for p in model.parameters():
        p.requires_grad = False
    for p in model.norm.parameters():
        p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Strategy        : Freeze ALL")
    print(f"Trainable params: {trainable:,} / {total:,}")
    return model

results_freeze_all = run_pipeline(
    build_fn      = build_model_freeze_all,
    save_path     = "/kaggle/working/best_freeze_all.pth",
    strategy_name = "Freeze ALL"
)

## Strategy 2 — Fine-tune ALL
All layers unfrozen. Every parameter is trainable.

In [ ]:
# MODEL — FINE-TUNE ALL
def build_model_finetune_all():
    model = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=True,
        num_classes=CONFIG["num_classes"]
    )
    for p in model.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Strategy        : Fine-tune ALL")
    print(f"Trainable params: {trainable:,}")
    return model

results_finetune_all = run_pipeline(
    build_fn      = build_model_finetune_all,
    save_path     = "/kaggle/working/best_finetune_all.pth",
    strategy_name = "Fine-tune ALL"
)

## Strategy 3 — 1-Fine-tune
Only the last Swin stage (layers[3]) + norm + head are unfrozen.

In [ ]:
# MODEL — 1-FINE-TUNE (last stage + norm + head)
def build_model_1_finetune():
    model = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=True,
        num_classes=CONFIG["num_classes"]
    )
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layers[3].parameters():
        p.requires_grad = True
    for p in model.norm.parameters():
        p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Strategy        : 1-Fine-tune")
    print(f"Trainable params: {trainable:,} / {total:,}")
    return model

results_1_finetune = run_pipeline(
    build_fn      = build_model_1_finetune,
    save_path     = "/kaggle/working/best_1_finetune.pth",
    strategy_name = "1-Fine-tune"
)

## Strategy 4 — 2-Fine-tune
Last two Swin stages (layers[2] + layers[3]) + norm + head are unfrozen.

In [ ]:
# MODEL — 2-FINE-TUNE (stages 3+4 + norm + head)
def build_model_2_finetune():
    model = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=True,
        num_classes=CONFIG["num_classes"]
    )
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layers[2].parameters():
        p.requires_grad = True
    for p in model.layers[3].parameters():
        p.requires_grad = True
    for p in model.norm.parameters():
        p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Strategy        : 2-Fine-tune")
    print(f"Trainable params: {trainable:,} / {total:,}")
    return model

results_2_finetune = run_pipeline(
    build_fn      = build_model_2_finetune,
    save_path     = "/kaggle/working/best_2_finetune.pth",
    strategy_name = "2-Fine-tune"
)

## Strategy 5 — 3-Fine-tune
Last three Swin stages (layers[1] + layers[2] + layers[3]) + norm + head are unfrozen.

In [ ]:
# MODEL — 3-FINE-TUNE (stages 2+3+4 + norm + head)
def build_model_3_finetune():
    model = timm.create_model(
        'swin_base_patch4_window7_224',
        pretrained=True,
        num_classes=CONFIG["num_classes"]
    )
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layers[1].parameters():
        p.requires_grad = True
    for p in model.layers[2].parameters():
        p.requires_grad = True
    for p in model.layers[3].parameters():
        p.requires_grad = True
    for p in model.norm.parameters():
        p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Strategy        : 3-Fine-tune")
    print(f"Trainable params: {trainable:,} / {total:,}")
    return model

results_3_finetune = run_pipeline(
    build_fn      = build_model_3_finetune,
    save_path     = "/kaggle/working/best_3_finetune.pth",
    strategy_name = "3-Fine-tune"
)

## Comparison of All Strategies

In [ ]:
# COMPARISON TABLE
all_results = [results_freeze_all, results_finetune_all, results_1_finetune,
               results_2_finetune, results_3_finetune]

df = pd.DataFrame(all_results).rename(columns={
    "strategy" : "Strategy",
    "accuracy" : "Test Accuracy (%)",
    "precision": "Test Precision (%)",
    "f1"       : "Test F1-Score (%)"
})
df = df.sort_values("Test F1-Score (%)", ascending=False).reset_index(drop=True)
df.index += 1
print("COMPARATIVE TABLE — ALL STRATEGIES (sorted by F1-Score)")
df

In [ ]:
# COMPARISON BAR CHART
strategies = [r["strategy"] for r in all_results]
accuracy   = [r["accuracy"]  for r in all_results]
precision  = [r["precision"] for r in all_results]
f1         = [r["f1"]        for r in all_results]
colors     = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#f39c12']

x     = np.arange(len(strategies))
width = 0.6

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Strategy Comparison — Test Set Results",
             fontsize=16, fontweight='bold', y=1.02)

for ax, values, title in zip(
    axes,
    [accuracy, precision, f1],
    ["Accuracy (%)", "Precision (%)", "F1-Score (%)"]
):
    bars = ax.bar(x, values, width, color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace(" ", "\n") for s in strategies], fontsize=9)
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/comparison_all_strategies.png', dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved.")